# 04 - Collaborative Filtering Item-Based

Recomenda produtos similares aos que o usuário já comprou, via similaridade de co-ocorrência entre itens (cosine sobre o histórico `prior`). Ver `docs/NOTEBOOKS.md` (seção 4).

## 0. Configuração Inicial

In [1]:
import json
import pickle
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path("..").resolve()))
from src.evaluation.metrics import evaluate_recommendations, pairs_to_ground_truth
from src.evaluation.ranking import recommendations_from_score_matrix

RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    """Fixa a seed de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models/item_based_cf")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open("../configs/model_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
K = config["evaluation"]["k"]

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("item_based_cf")

print(f"K={K}")

K=10


## 1. Carregamento dos Artefatos de Preprocessing

In [2]:
with open(PROCESSED_DIR / "vocabularies.pkl", "rb") as f:
    vocab = pickle.load(f)

interactions_prior = sp.load_npz(PROCESSED_DIR / "interactions_prior.npz")
val_pairs = pd.read_pickle(PROCESSED_DIR / "val_pairs.pkl")
test_pairs = pd.read_pickle(PROCESSED_DIR / "test_pairs.pkl")

n_users, n_items = interactions_prior.shape
val_ground_truth = pairs_to_ground_truth(val_pairs)
test_ground_truth = pairs_to_ground_truth(test_pairs)

print(f"n_users={n_users:,} | n_items={n_items:,}")

n_users=126,408 | n_items=3,000


## 2. Similaridade Item-Item (Cosine sobre Co-ocorrência)

In [3]:
item_similarity = cosine_similarity(interactions_prior.T, dense_output=True)
item_similarity = item_similarity.astype(np.float32)

print(f"Matriz de similaridade item-item: {item_similarity.shape}")

Matriz de similaridade item-item: (3000, 3000)


## 3. Random Search de top_m Vizinhos (Validação)

A matriz de similaridade completa (3.000x3.000) inclui muitas similaridades pequenas e ruidosas. Busca aleatória sobre `top_m` (quantos vizinhos mais similares manter por item, zerando o restante) definida em `configs/model_config.yaml`, avaliada no split de **validação**. `top_m = 3000` equivale a manter a matriz completa (sem truncamento).

In [4]:
def truncate_similarity(similarity: np.ndarray, top_m: int) -> np.ndarray:
    """Mantem apenas os top_m maiores valores por linha, zera os demais.

    A diagonal (auto-similaridade = 1.0) sempre permanece, pois e o maior
    valor possivel em cada linha.

    Args:
        similarity: matriz de similaridade item-item (n_items, n_items).
        top_m: numero de vizinhos mais similares mantidos por item.

    Returns:
        Matriz de similaridade truncada (mesma forma).
    """
    if top_m >= similarity.shape[1]:
        return similarity
    truncated = similarity.copy()
    for i in range(truncated.shape[0]):
        row = truncated[i]
        threshold = np.partition(row, -top_m)[-top_m]
        row[row < threshold] = 0.0
    return truncated


def compute_topk_recommendations(
    user_indices: list[int],
    interactions: sp.csr_matrix,
    similarity: np.ndarray,
    k: int,
    batch_size: int = 2000,
) -> dict[int, list[int]]:
    """Gera o top-k de recomendacoes por similaridade, processando em lotes.

    Args:
        user_indices: Lista de user_idx avaliados.
        interactions: Matriz esparsa (n_users, n_items) de historico de compras.
        similarity: Matriz de similaridade item-item (n_items, n_items).
        k: Tamanho do top-k recomendado.
        batch_size: Numero de usuarios processados por lote (controle de memoria).

    Returns:
        Mapeamento user_idx -> lista de item_idx recomendados, ordenada por score.
    """
    recommendations: dict[int, list[int]] = {}
    for start in range(0, len(user_indices), batch_size):
        batch = user_indices[start : start + batch_size]
        batch_scores = interactions[batch].dot(similarity)
        recommendations.update(
            recommendations_from_score_matrix(batch, batch_scores, k)
        )
    return recommendations


search_cfg = config["item_based_cf"]["search"]
candidates = search_cfg["top_m_choices"]
n_trials = min(search_cfg["n_trials"], len(candidates))
search_rng = np.random.default_rng(RANDOM_SEED)
trial_top_m = search_rng.choice(candidates, size=n_trials, replace=False)

search_results = []
for trial_idx, top_m in enumerate(trial_top_m):
    top_m = int(top_m)
    trial_similarity = truncate_similarity(item_similarity, top_m)
    trial_recs = compute_topk_recommendations(
        list(val_ground_truth.keys()), interactions_prior, trial_similarity, K
    )
    trial_metrics = evaluate_recommendations(trial_recs, val_ground_truth, K)
    search_results.append({"top_m": top_m, **trial_metrics})

    with mlflow.start_run(run_name=f"search_trial_{trial_idx}_topm_{top_m}"):
        mlflow.log_param("top_m", top_m)
        mlflow.log_metrics(
            {f"val_{name}": value for name, value in trial_metrics.items()}
        )

search_df = pd.DataFrame(search_results).sort_values("recall_at_k", ascending=False)
TOP_M = int(search_df.iloc[0]["top_m"])
item_similarity = truncate_similarity(item_similarity, TOP_M)
print(search_df)
print(f"Melhor top_m (validacao): {TOP_M}")

   top_m  precision_at_k  recall_at_k  ndcg_at_k  map_at_k
2   3000        0.130679     0.222148   0.217030  0.127153
1    300        0.126491     0.212607   0.211131  0.123375
3    100        0.122124     0.204358   0.205289  0.119905
0     50        0.114941     0.193549   0.194751  0.113043
Melhor top_m (validacao): 3000


## 4. Geração de Recomendações em Lotes (Melhor top_m)

In [5]:
val_recommendations = compute_topk_recommendations(
    list(val_ground_truth.keys()), interactions_prior, item_similarity, K
)
test_recommendations = compute_topk_recommendations(
    list(test_ground_truth.keys()), interactions_prior, item_similarity, K
)

## 5. Avaliação

In [6]:
val_metrics = evaluate_recommendations(val_recommendations, val_ground_truth, K)
test_metrics = evaluate_recommendations(test_recommendations, test_ground_truth, K)

print("Validação:", val_metrics)
print("Teste:", test_metrics)

Validação: {'precision_at_k': 0.13067876166868836, 'recall_at_k': 0.22214801462874315, 'ndcg_at_k': 0.21703016080040424, 'map_at_k': 0.12715331866338098}
Teste: {'precision_at_k': 0.1292426959181521, 'recall_at_k': 0.2236194213956057, 'ndcg_at_k': 0.21719785440370326, 'map_at_k': 0.12744316001352002}


## 6. Persistência e Rastreamento no MLflow

In [7]:
with open(PROCESSED_DIR / "split_meta.json", encoding="utf-8") as f:
    split_meta = json.load(f)

sp.save_npz(MODELS_DIR / "item_similarity.npz", sp.csr_matrix(item_similarity))

metrics_payload = {
    "k": K,
    "top_m": TOP_M,
    "search_results": search_results,
    "validation": val_metrics,
    "test": test_metrics,
}
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

with mlflow.start_run(run_name="item_based_cf_final"):
    mlflow.log_param("k", K)
    mlflow.log_param("similarity_metric", "cosine")
    mlflow.log_param("top_m", TOP_M)
    mlflow.log_param("dataset_hash", split_meta["dataset_hash"])
    mlflow.log_param("selected_via_random_search", True)
    mlflow.log_metrics({f"val_{name}": value for name, value in val_metrics.items()})
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})
    mlflow.log_artifact(str(MODELS_DIR / "metrics.json"))

print("Item-based CF avaliado e rastreado no MLflow.")

Item-based CF avaliado e rastreado no MLflow.
